# AE3 — Transformación y enriquecimiento del dataset de abandono de clientes

**Proyecto integrador · Etapa 2: transformación y enriquecimiento de datos**

Flujo de este notebook:

```
dataset de AE2 → transformación y validación → enriquecimiento → respuestas a las preguntas
              → tablas de resultados (.csv) + dataset preparado (.csv) + resumen descriptivo (.txt)
```

Se continúa con el mismo proyecto, dataset y repositorio de la AE2. Ejecuta las celdas en orden (**Kernel → Restart & Run All**).

## 1. Necesidad, preguntas de negocio y variables

**Necesidad (escenario hipotético, definida en AE2):** el equipo de retención de una compañía de telecomunicaciones
necesita saber qué perfiles de cliente concentran las cancelaciones del servicio, para orientar sus campañas de fidelización.

**Pregunta general (AE2):** ¿cómo varía la proporción de clientes que abandonaron el servicio según el tipo de contrato,
la antigüedad, el cargo mensual, el método de pago y el tipo de servicio de internet?

**Ajuste justificado:** la pregunta general de AE2 se **desglosa** en seis preguntas de negocio, una por cada componente del
diseño inicial (Tabla 3 del documento AE2). No cambia el alcance: P1 a P5 corresponden a los componentes de la pregunta y
P6 al «ranking de segmentos» previsto en AE2 para comunicar resultados. Además, se agrega el **ingreso mensual perdido**
(suma del cargo mensual de quienes abandonaron) porque es un dato que el equipo de retención puede usar para priorizar
y se calcula solo con las variables existentes.

| # | Pregunta de negocio | Variables (nombre preparado) | Resultado esperado |
|---|---|---|---|
| P1 | ¿Qué proporción de clientes abandonó y cuánto ingreso mensual representa? | `abandono`, `cargo_mensual` | Tasa general de abandono e ingreso mensual perdido |
| P2 | ¿Cómo varía el abandono según el tipo de contrato? | `tipo_contrato`, `abandono` | Tasa y participación por tipo de contrato |
| P3 | ¿Cómo varía el abandono según la antigüedad del cliente? | `antiguedad_meses` → `rango_antiguedad` | Tasa por rango de antigüedad |
| P4 | ¿Se diferencian en sus cargos los clientes que abandonan? | `cargo_mensual` → `rango_cargo_mensual`, `cargo_total` | Tasa por nivel de cargo y comparación de cargos |
| P5 | ¿Hay diferencias según el método de pago y el servicio de internet? | `metodo_pago`, `pago_automatico`, `servicio_internet` | Tasa por método de pago y por servicio |
| P6 | ¿Qué segmentos (contrato × internet) concentran el abandono? | `tipo_contrato`, `servicio_internet` | Ranking de segmentos para el equipo de retención |

**Limitaciones conocidas:** los datos son de una compañía **ficticia** (IBM Sample Datasets), no hay fechas (no se pueden
ver tendencias en el tiempo), no hay costos (no se puede hablar de rentabilidad) y el análisis es descriptivo (no explica causas).

## 2. Preparar el entorno y cargar el dataset

In [1]:
import sys
import warnings
from pathlib import Path

# Buscamos la carpeta raíz del proyecto (la que contiene src/clases_po).
ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / "src" / "clases_po").exists():
    ROOT = ROOT.parent
# Agregamos "src" a las rutas de Python para poder importar nuestras clases.
sys.path.insert(0, str(ROOT / "src"))

# Oculta advertencias informativas de las librerías (no afectan el resultado).
warnings.filterwarnings("ignore")
from huggingface_hub.utils import logging as hf_logging
hf_logging.set_verbosity_error()

import pandas as pd
from clases_po import HFDatasetLoader, Reportes, tabla_abandono

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 200)

# Carpetas de salida de esta entrega (rutas relativas a la raíz del proyecto).
CARPETA_DATOS = ROOT / "AE3" / "datos"
CARPETA_TABLAS = ROOT / "AE3" / "resultados" / "tablas"
CARPETA_DATOS.mkdir(parents=True, exist_ok=True)
CARPETA_TABLAS.mkdir(parents=True, exist_ok=True)
print("Carpetas de salida listas.")

Carpetas de salida listas.


In [2]:
# Descargamos el mismo dataset y split de AE2.
loader = HFDatasetLoader(dataset_name="scikit-learn/churn-prediction", split="train")
df_original = loader.load()

# Guardamos una copia sin modificar para comparar al final (df_original no se toca).
df = df_original.copy()
loader.info()

{'dataset': 'scikit-learn/churn-prediction',
 'config': None,
 'split': 'train',
 'muestra': 'no (split completo)',
 'filas': 7043,
 'columnas': 21}

## 3. Transformación y validación

Cada paso indica **qué cambia, por qué y cuántos registros afecta**. Al final se compara el antes y el después.

En la lista `bitacora` se registra cada decisión; esa lista se exporta después como tabla.

In [3]:
# Aquí anotamos cada transformación: paso, qué se hizo, registros afectados y justificación.
bitacora = []

def registrar(paso, accion, afectados, justificacion):
    """Agrega una fila a la bitácora de transformaciones."""
    bitacora.append({"paso": paso, "accion": accion,
                     "registros_afectados": int(afectados), "justificacion": justificacion})

### 3.1 Corregir el tipo de `TotalCharges` (texto → número)

In [4]:
# TotalCharges viene como texto. Quitamos espacios y convertimos a número;
# lo que no se pueda convertir (los blancos) queda como vacío (NaN).
total_numerico = pd.to_numeric(df["TotalCharges"].str.strip(), errors="coerce")
no_convertibles = total_numerico.isna()
print("Valores que no se pudieron convertir:", int(no_convertibles.sum()))

# ¿Quiénes son? Revisamos su antigüedad y si abandonaron.
print(df.loc[no_convertibles, ["tenure", "Churn"]].value_counts())

df["TotalCharges"] = total_numerico
registrar("3.1", "TotalCharges convertido de texto a número (float)", len(df),
          "Es un monto; como texto no permite sumar ni promediar.")

Valores que no se pudieron convertir: 11
tenure  Churn
0       No       11
Name: count, dtype: int64


### 3.2 Tratar los valores faltantes de `TotalCharges`

Los 11 vacíos corresponden **todos** a clientes con antigüedad 0 meses (recién llegados, todavía sin facturación acumulada).
**Criterio:** se reemplazan por **0**, porque un cliente con 0 meses aún no acumula cargos. Para no ocultar la decisión,
se crea la columna `cargo_total_imputado` que marca esos 11 registros. No se eliminan filas: son clientes reales del
dataset y cuentan para la tasa de abandono.

In [5]:
# Comprobamos el criterio antes de aplicarlo: todos los vacíos deben tener antigüedad 0.
assert (df.loc[no_convertibles, "tenure"] == 0).all()

df["cargo_total_imputado"] = no_convertibles          # True en los 11 registros tratados
df["TotalCharges"] = df["TotalCharges"].fillna(0.0)   # vacío -> 0
print("Nulos en TotalCharges después:", int(df["TotalCharges"].isna().sum()))
registrar("3.2", "TotalCharges vacío reemplazado por 0 y marcado en cargo_total_imputado",
          no_convertibles.sum(), "Los 11 casos tienen antigüedad 0: aún no acumulan cargos.")

Nulos en TotalCharges después: 0


### 3.3 Revisar faltantes, duplicados y espacios en el resto de columnas

In [6]:
# Valores faltantes en todo el DataFrame.
print("Valores nulos totales:", int(df.isna().sum().sum()))

# Duplicados: por fila completa, por ID y por atributos (sin el ID).
print("Filas duplicadas completas:", int(df.duplicated().sum()))
print("customerID repetidos:", int(df["customerID"].duplicated().sum()))
dup_atributos = df.drop(columns="customerID").duplicated()
print("Filas con atributos idénticos a otra (ignorando el ID):", int(dup_atributos.sum()))

# Espacios al inicio o al final en las columnas de texto.
# (is_string_dtype detecta texto tanto en pandas 2 como en pandas 3)
columnas_texto = [c for c in df.columns if pd.api.types.is_string_dtype(df[c])]
con_espacios = {c: int((df[c] != df[c].str.strip()).sum()) for c in columnas_texto}
print("Valores con espacios sobrantes:", sum(con_espacios.values()))

Valores nulos totales: 0
Filas duplicadas completas: 0
customerID repetidos: 0
Filas con atributos idénticos a otra (ignorando el ID): 22
Valores con espacios sobrantes: 0


**Resultado:** no quedan nulos, no hay filas ni IDs duplicados y no quedan espacios sobrantes (los únicos eran los
blancos de `TotalCharges`, ya tratados). Hay 22 filas con los mismos atributos que otra pero **con distinto `customerID`**.
**No se eliminan**, porque son clientes distintos con el mismo perfil (por ejemplo, clientes nuevos con el plan básico),
y eliminarlos alteraría los conteos.

In [7]:
registrar("3.3", "Control de nulos, duplicados (fila, ID y atributos) y espacios", 0,
          "Sin problemas; 22 perfiles idénticos con distinto ID se conservan por ser clientes distintos.")

### 3.4 Revisar la coherencia entre columnas

In [8]:
# 1) "No internet service" debe aparecer solo cuando el cliente no tiene internet.
servicios_internet = ["OnlineSecurity", "OnlineBackup", "DeviceProtection",
                      "TechSupport", "StreamingTV", "StreamingMovies"]
sin_internet = df["InternetService"] == "No"
incoherentes_internet = sum(int(((df[c] == "No internet service") != sin_internet).sum())
                            for c in servicios_internet)

# 2) "No phone service" debe aparecer solo cuando el cliente no tiene teléfono.
incoherentes_telefono = int(((df["MultipleLines"] == "No phone service")
                             != (df["PhoneService"] == "No")).sum())

# 3) El cargo total debería parecerse a cargo mensual x antigüedad.
con_antiguedad = df["tenure"] > 0
razon = df.loc[con_antiguedad, "TotalCharges"] / (df.loc[con_antiguedad, "MonthlyCharges"]
                                                  * df.loc[con_antiguedad, "tenure"])
total_menor_que_mensual = int((df.loc[con_antiguedad, "TotalCharges"]
                               < df.loc[con_antiguedad, "MonthlyCharges"] - 0.01).sum())

print("Incoherencias servicio de internet:", incoherentes_internet)
print("Incoherencias servicio telefónico:", incoherentes_telefono)
print("Cargo total menor que un mes de cargo:", total_menor_que_mensual)
print("Razón cargo_total / (cargo_mensual x antigüedad):")
print(razon.describe().round(3))

Incoherencias servicio de internet: 0
Incoherencias servicio telefónico: 0
Cargo total menor que un mes de cargo: 0
Razón cargo_total / (cargo_mensual x antigüedad):
count    7032.000
mean        1.000
std         0.051
min         0.689
25%         0.980
50%         1.000
75%         1.020
max         1.573
dtype: float64


**Resultado:** no hay incoherencias entre servicios. El cargo total coincide en promedio con cargo mensual × antigüedad
(razón media ≈ 1,00), con variaciones que se explican por cambios de tarifa en el tiempo. Se informa el control y no se modifica nada.

In [9]:
registrar("3.4", "Control de coherencia entre servicios y entre cargo total y mensual", 0,
          "Sin incoherencias; las diferencias de cargo se explican por cambios de tarifa.")

### 3.5 Normalizar nombres de columnas y categorías

- Columnas en español y `snake_case` (por ejemplo, `MonthlyCharges` → `cargo_mensual`).
- `Yes`/`No` → `Sí`/`No`, y `SeniorCitizen` 0/1 → `No`/`Sí`, para que todas las variables binarias usen el mismo formato.
- `No internet service` y `No phone service` → `Sin servicio`: es la misma idea escrita de dos formas.
- Contrato, internet y método de pago traducidos al español. El contrato queda como categoría **ordenada** (de menor a mayor compromiso).

In [10]:
# Nuevos nombres de columnas.
nombres = {
    "customerID": "id_cliente", "gender": "genero", "SeniorCitizen": "adulto_mayor",
    "Partner": "tiene_pareja", "Dependents": "tiene_dependientes", "tenure": "antiguedad_meses",
    "PhoneService": "servicio_telefonico", "MultipleLines": "lineas_multiples",
    "InternetService": "servicio_internet", "OnlineSecurity": "seguridad_en_linea",
    "OnlineBackup": "respaldo_en_linea", "DeviceProtection": "proteccion_dispositivos",
    "TechSupport": "soporte_tecnico", "StreamingTV": "tv_streaming",
    "StreamingMovies": "peliculas_streaming", "Contract": "tipo_contrato",
    "PaperlessBilling": "factura_electronica", "PaymentMethod": "metodo_pago",
    "MonthlyCharges": "cargo_mensual", "TotalCharges": "cargo_total", "Churn": "abandono",
}
df = df.rename(columns=nombres)

# Traducción de los valores Sí/No y de "sin servicio".
si_no = {"Yes": "Sí", "No": "No", "No internet service": "Sin servicio", "No phone service": "Sin servicio"}
columnas_si_no = ["tiene_pareja", "tiene_dependientes", "servicio_telefonico", "lineas_multiples",
                  "seguridad_en_linea", "respaldo_en_linea", "proteccion_dispositivos", "soporte_tecnico",
                  "tv_streaming", "peliculas_streaming", "factura_electronica", "abandono"]
for columna in columnas_si_no:
    df[columna] = df[columna].map(si_no)

# adulto_mayor viene como 0/1: lo pasamos al mismo formato Sí/No.
df["adulto_mayor"] = df["adulto_mayor"].map({1: "Sí", 0: "No"})

# Traducción de contrato, internet y método de pago.
df["tipo_contrato"] = df["tipo_contrato"].map(
    {"Month-to-month": "Mes a mes", "One year": "Un año", "Two year": "Dos años"})
df["servicio_internet"] = df["servicio_internet"].map(
    {"DSL": "DSL", "Fiber optic": "Fibra óptica", "No": "Sin internet"})
df["metodo_pago"] = df["metodo_pago"].map({
    "Electronic check": "Cheque electrónico", "Mailed check": "Cheque por correo",
    "Bank transfer (automatic)": "Transferencia bancaria (automática)",
    "Credit card (automatic)": "Tarjeta de crédito (automática)"})

# Validación: si algún valor no estaba en los diccionarios, quedaría vacío.
print("Valores sin traducir (deben ser 0):", int(df.isna().sum().sum()))
assert df.isna().sum().sum() == 0

Valores sin traducir (deben ser 0): 0


In [11]:
# Convertimos las columnas de texto con pocos valores a tipo "category" (ocupan menos y conservan el orden).
orden_contrato = ["Mes a mes", "Un año", "Dos años"]
df["tipo_contrato"] = pd.Categorical(df["tipo_contrato"], categories=orden_contrato, ordered=True)
# (is_string_dtype detecta texto tanto en pandas 2 como en pandas 3)
for columna in df.columns:
    if columna != "id_cliente" and pd.api.types.is_string_dtype(df[columna]):
        df[columna] = df[columna].astype("category")

# Redondeamos los montos a 2 decimales.
df[["cargo_mensual", "cargo_total"]] = df[["cargo_mensual", "cargo_total"]].round(2)

registrar("3.5", "Columnas renombradas a español y categorías normalizadas (Sí/No, Sin servicio, traducciones)",
          len(df), "Formato uniforme y legible para las tablas y gráficas de AE4.")
df.dtypes.value_counts()

category    7
category    6
float64     2
str         1
category    1
int64       1
category    1
category    1
category    1
bool        1
Name: count, dtype: int64

### 3.6 Seleccionar columnas y registros pertinentes

- **Registros:** se conservan **todos** (7 043). El alcance son todos los clientes del dataset y no hay registros inválidos.
- **Columnas:** se elimina `genero`. Ninguna pregunta de negocio la usa, y el equipo de retención no orienta campañas por
  género. Las demás columnas se conservan porque se usan en las preguntas o en los indicadores derivados.

In [12]:
df = df.drop(columns=["genero"])
registrar("3.6", "Se elimina la columna genero; se conservan los 7 043 registros", 0,
          "genero no responde ninguna pregunta de negocio; no hay registros fuera del alcance.")
print("Dimensiones después de la selección:", df.shape)

Dimensiones después de la selección: (7043, 21)


## 4. Enriquecimiento: variables derivadas

In [13]:
# 1) abandono_num: 1 si el cliente abandonó, 0 si no. Sirve para sumar abandonos y calcular tasas.
df["abandono_num"] = (df["abandono"] == "Sí").astype(int)

# 2) rango_antiguedad: agrupa los meses de antigüedad en tramos de 12 meses (años de permanencia).
cortes = [-1, 12, 24, 36, 48, 60, 72]
etiquetas = ["0-12 meses", "13-24 meses", "25-36 meses", "37-48 meses", "49-60 meses", "61-72 meses"]
df["rango_antiguedad"] = pd.cut(df["antiguedad_meses"], bins=cortes, labels=etiquetas)

# 3) rango_cargo_mensual: divide a los clientes en 4 grupos de tamaño parecido según su cargo mensual (cuartiles).
cuartiles = df["cargo_mensual"].quantile([0, 0.25, 0.5, 0.75, 1]).round(2).tolist()
nombres_cuartil = ["Q1 Bajo", "Q2 Medio-bajo", "Q3 Medio-alto", "Q4 Alto"]
etiquetas_cargo = [f"{n} ({cuartiles[i]:.2f}-{cuartiles[i + 1]:.2f})" for i, n in enumerate(nombres_cuartil)]
df["rango_cargo_mensual"] = pd.qcut(df["cargo_mensual"], q=4, labels=etiquetas_cargo)

# 4) num_servicios_adicionales: cuántos de los 6 servicios extra tiene contratados (0 a 6).
adicionales = ["seguridad_en_linea", "respaldo_en_linea", "proteccion_dispositivos",
               "soporte_tecnico", "tv_streaming", "peliculas_streaming"]
df["num_servicios_adicionales"] = (df[adicionales] == "Sí").sum(axis=1)

# 5) pago_automatico: Sí si paga con transferencia o tarjeta automática.
df["pago_automatico"] = (df["metodo_pago"].astype(str).str.contains("automática")
                         .map({True: "Sí", False: "No"}).astype("category"))

# 6) ingreso_mensual_perdido: cargo mensual que deja de recibirse si el cliente abandonó (0 si sigue).
df["ingreso_mensual_perdido"] = df["cargo_mensual"].where(df["abandono_num"] == 1, 0.0)

# 7) segmento: combinación de tipo de contrato y servicio de internet (para el ranking de P6).
df["segmento"] = (df["tipo_contrato"].astype(str) + " | " + df["servicio_internet"].astype(str)).astype("category")

registrar("4", "Se agregan 7 variables derivadas (abandono_num, rangos, servicios, pago automático, ingreso perdido, segmento)",
          len(df), "Permiten calcular tasas, comparar por tramos y priorizar segmentos.")
df[["antiguedad_meses", "rango_antiguedad", "cargo_mensual", "rango_cargo_mensual",
    "num_servicios_adicionales", "pago_automatico", "abandono", "abandono_num",
    "ingreso_mensual_perdido", "segmento"]].head()

,antiguedad_meses,rango_antiguedad,cargo_mensual,rango_cargo_mensual,num_servicios_adicionales,pago_automatico,abandono,abandono_num,ingreso_mensual_perdido,segmento
0,1,0-12 meses,29.85,Q1 Bajo (18.25-35.50),1,No,No,0,0.00,Mes a mes | DSL
1,34,25-36 meses,56.95,Q2 Medio-bajo (35.50-70.35),2,No,No,0,0.00,Un año | DSL
2,2,0-12 meses,53.85,Q2 Medio-bajo (35.50-70.35),2,No,Sí,1,53.85,Mes a mes | DSL
3,45,37-48 meses,42.30,Q2 Medio-bajo (35.50-70.35),3,Sí,No,0,0.00,Un año | DSL
4,2,0-12 meses,70.70,Q3 Medio-alto (70.35-89.85),0,No,Sí,1,70.70,Mes a mes | Fibra óptica


In [14]:
# Validaciones del enriquecimiento:
assert df["abandono_num"].sum() == (df_original["Churn"] == "Yes").sum()   # mismos abandonos que el original
assert df["rango_antiguedad"].notna().all()                                # todos los meses quedaron en un rango
assert df["rango_cargo_mensual"].notna().all()                             # todos los cargos quedaron en un cuartil
assert df["num_servicios_adicionales"].between(0, 6).all()
print("Clientes por rango de antigüedad:")
print(df["rango_antiguedad"].value_counts(sort=False).to_string())
print("\nClientes por rango de cargo mensual:")
print(df["rango_cargo_mensual"].value_counts(sort=False).to_string())

Clientes por rango de antigüedad:
rango_antiguedad
0-12 meses     2186
13-24 meses    1024
25-36 meses     832
37-48 meses     762
49-60 meses     832
61-72 meses    1407

Clientes por rango de cargo mensual:
rango_cargo_mensual
Q1 Bajo (18.25-35.50)          1762
Q2 Medio-bajo (35.50-70.35)    1766
Q3 Medio-alto (70.35-89.85)    1757
Q4 Alto (89.85-118.75)         1758


| Variable nueva | Regla / fórmula | Utilidad |
|---|---|---|
| `abandono_num` | 1 si `abandono` = Sí; 0 si No | Sumar abandonos y calcular tasas (P1–P6) |
| `rango_antiguedad` | Tramos de 12 meses sobre `antiguedad_meses` | Comparar el abandono por tiempo de permanencia (P3) |
| `rango_cargo_mensual` | Cuartiles de `cargo_mensual` (4 grupos de ~25 %) | Comparar el abandono por nivel de cargo (P4) |
| `num_servicios_adicionales` | Conteo de «Sí» en los 6 servicios extra | Describir el nivel de servicios del cliente |
| `pago_automatico` | Sí si `metodo_pago` es automático | Agrupar los métodos de pago (P5) |
| `ingreso_mensual_perdido` | `cargo_mensual` si abandonó; 0 si no | Cuantificar el impacto del abandono (P1–P6) |
| `segmento` | `tipo_contrato` + « \| » + `servicio_internet` | Ranking de segmentos (P6) |
| `cargo_total_imputado` | True si `cargo_total` estaba vacío (paso 3.2) | Trazabilidad de la imputación |

## 5. Comparación antes y después

In [15]:
# Comparamos tamaño, tipos y nulos entre el DataFrame original y el preparado.
comparacion = pd.DataFrame({
    "original": [df_original.shape[0], df_original.shape[1],
                 int(df_original.isna().sum().sum()),
                 int((df_original["TotalCharges"].str.strip() == "").sum()),
                 ", ".join(f"{t}: {n}" for t, n in df_original.dtypes.astype(str).value_counts().items())],
    "preparado": [df.shape[0], df.shape[1], int(df.isna().sum().sum()), 0,
                  ", ".join(f"{t}: {n}" for t, n in df.dtypes.astype(str).value_counts().items())],
}, index=["filas", "columnas", "valores nulos", "TotalCharges en blanco", "tipos de columnas"])
comparacion

,original,preparado
filas,7043,7043
columnas,21,28
valores nulos,0,0
TotalCharges en blanco,11,0
tipos de columnas,"str: 18, int64: 2, float64: 1","category: 20, int64: 3, float64: 3, str: 1, bo..."


In [16]:
# Bitácora completa de transformaciones (también se exporta como CSV).
tabla_bitacora = pd.DataFrame(bitacora)
tabla_bitacora

,paso,accion,registros_afectados,justificacion
0,3.1,TotalCharges convertido de texto a número (float),7043,Es un monto; como texto no permite sumar ni pr...
1,3.2,TotalCharges vacío reemplazado por 0 y marcado...,11,Los 11 casos tienen antigüedad 0: aún no acumu...
2,3.3,"Control de nulos, duplicados (fila, ID y atrib...",0,Sin problemas; 22 perfiles idénticos con disti...
3,3.4,Control de coherencia entre servicios y entre ...,0,Sin incoherencias; las diferencias de cargo se...
4,3.5,Columnas renombradas a español y categorías no...,7043,Formato uniforme y legible para las tablas y g...
5,3.6,Se elimina la columna genero; se conservan los...,0,genero no responde ninguna pregunta de negocio...
6,4,Se agregan 7 variables derivadas (abandono_num...,7043,"Permiten calcular tasas, comparar por tramos y..."


## 6. Respuestas a las preguntas de negocio

Todas las tablas usan la función `tabla_abandono` (en `src/clases_po/indicadores.py`), que calcula por grupo:

| Columna | Significado | Unidad |
|---|---|---|
| `clientes` | Clientes del grupo | clientes |
| `abandonos` | Clientes del grupo que abandonaron | clientes |
| `tasa_abandono_pct` | abandonos ÷ clientes del grupo × 100 | % |
| `participacion_clientes_pct` | clientes del grupo ÷ total de clientes × 100 | % |
| `participacion_abandonos_pct` | abandonos del grupo ÷ total de abandonos × 100 | % |
| `cargo_mensual_promedio` | Promedio del cargo mensual del grupo | unidades monetarias/mes* |
| `ingreso_mensual_perdido` | Suma del cargo mensual de quienes abandonaron | unidades monetarias/mes* |
| `participacion_ingreso_perdido_pct` | ingreso perdido del grupo ÷ ingreso perdido total × 100 | % |

\* La ficha del dataset no documenta la moneda.

In [17]:
# Diccionario donde guardamos cada tabla de resultados para exportarlas al final.
tablas = {}

### P1. ¿Qué proporción de clientes abandonó y cuánto ingreso mensual representa?

In [18]:
p1 = pd.DataFrame({
    "clientes": [len(df)],
    "abandonos": [int(df["abandono_num"].sum())],
    "tasa_abandono_pct": [round(df["abandono_num"].mean() * 100, 2)],
    "ingreso_mensual_total": [round(df["cargo_mensual"].sum(), 2)],
    "ingreso_mensual_perdido": [round(df["ingreso_mensual_perdido"].sum(), 2)],
})
p1["ingreso_perdido_pct"] = (p1["ingreso_mensual_perdido"] / p1["ingreso_mensual_total"] * 100).round(2)
tablas["P1_resumen_general"] = p1
p1

,clientes,abandonos,tasa_abandono_pct,ingreso_mensual_total,ingreso_mensual_perdido,ingreso_perdido_pct
0,7043,1869,26.54,456116.6,139130.85,30.5


**Respuesta P1:** 1 869 de 7 043 clientes abandonaron el servicio en el último mes, una **tasa de abandono del 26,54 %**
(aproximadamente 1 de cada 4 clientes). Esos clientes pagaban en conjunto 139 130,85 unidades monetarias al mes, el
**30,50 % del ingreso mensual** de todos los clientes (456 116,60).

**Utilidad:** el ingreso perdido pesa más que la proporción de clientes (30,5 % frente a 26,5 %). Esto indica que se van,
en promedio, clientes con cargos más altos (se confirma en P4).
**Limitación:** es una foto de un solo mes; sin fechas no se sabe si la tasa sube o baja con el tiempo.

### P2. ¿Cómo varía el abandono según el tipo de contrato?

In [19]:
p2 = tabla_abandono(df, "tipo_contrato")
tablas["P2_abandono_por_contrato"] = p2
p2

,tipo_contrato,clientes,abandonos,tasa_abandono_pct,participacion_clientes_pct,participacion_abandonos_pct,cargo_mensual_promedio,ingreso_mensual_perdido,participacion_ingreso_perdido_pct
0,Mes a mes,3875,1655,42.71,55.02,88.55,66.40,120847.10,86.86
1,Un año,1473,166,11.27,20.91,8.88,65.05,14118.45,10.15
2,Dos años,1695,48,2.83,24.07,2.57,60.77,4165.30,2.99


**Respuesta P2:** el contrato **mes a mes** concentra el abandono. Su tasa es del **42,71 %**, frente al 11,27 % del contrato a un año
y el 2,83 % del de dos años. Aunque son el 55,02 % de los clientes, aportan el **88,55 % de los abandonos** y el 86,86 % del ingreso
mensual perdido.

**Utilidad:** convertir clientes mes a mes a contratos de plazo es la línea de acción más relevante para el equipo de retención.
**Limitación:** el resultado describe una asociación. No demuestra que el contrato largo *cause* la permanencia: los clientes
más fieles pueden ser justamente los que eligen contratos largos.

### P3. ¿Cómo varía el abandono según la antigüedad del cliente?

In [20]:
p3 = tabla_abandono(df, "rango_antiguedad")
tablas["P3_abandono_por_antiguedad"] = p3
p3

,rango_antiguedad,clientes,abandonos,tasa_abandono_pct,participacion_clientes_pct,participacion_abandonos_pct,cargo_mensual_promedio,ingreso_mensual_perdido,participacion_ingreso_perdido_pct
0,0-12 meses,2186,1037,47.44,31.04,55.48,56.10,68954.25,49.56
1,13-24 meses,1024,294,28.71,14.54,15.73,61.36,23081.65,16.59
2,25-36 meses,832,180,21.63,11.81,9.63,65.58,15167.95,10.90
3,37-48 meses,762,145,19.03,10.82,7.76,66.32,12294.55,8.84
4,49-60 meses,832,120,14.42,11.81,6.42,70.55,10581.90,7.61
5,61-72 meses,1407,93,6.61,19.98,4.98,75.95,9050.55,6.51


**Respuesta P3:** el abandono baja a medida que aumenta la antigüedad. Es del **47,44 % en el primer año (0–12 meses)**, del 28,71 %
en el segundo y sigue bajando hasta el **6,61 % en los clientes de 61–72 meses**. Los clientes del primer año son el 31,04 % del
total, pero aportan el **55,48 % de los abandonos**.

**Utilidad:** el primer año es el periodo crítico; las acciones de bienvenida y acompañamiento temprano tendrían el mayor alcance.
**Limitación:** la antigüedad se mide al momento del corte. No se observa la misma cohorte en el tiempo, y el rango «0–12 meses»
incluye los 11 clientes con 0 meses.

### P4. ¿Se diferencian en sus cargos los clientes que abandonan?

In [21]:
# P4a: tasa de abandono por nivel de cargo mensual (cuartiles).
p4a = tabla_abandono(df, "rango_cargo_mensual")
tablas["P4a_abandono_por_cargo_mensual"] = p4a
p4a

,rango_cargo_mensual,clientes,abandonos,tasa_abandono_pct,participacion_clientes_pct,participacion_abandonos_pct,cargo_mensual_promedio,ingreso_mensual_perdido,participacion_ingreso_perdido_pct
0,Q1 Bajo (18.25-35.50),1762,198,11.24,25.02,10.59,22.23,4775.20,3.43
1,Q2 Medio-bajo (35.50-70.35),1766,434,24.58,25.07,23.22,55.73,23970.25,17.23
2,Q3 Medio-alto (70.35-89.85),1757,659,37.51,24.95,35.26,80.33,52881.65,38.01
3,Q4 Alto (89.85-118.75),1758,578,32.88,24.96,30.93,100.90,57503.75,41.33


In [22]:
# P4b: comparación directa de cargos y antigüedad entre quienes abandonaron y quienes permanecieron.
p4b = (df.groupby("abandono", observed=True)
         .agg(clientes=("id_cliente", "size"),
              cargo_mensual_promedio=("cargo_mensual", "mean"),
              cargo_mensual_mediana=("cargo_mensual", "median"),
              cargo_total_promedio=("cargo_total", "mean"),
              cargo_total_mediana=("cargo_total", "median"),
              antiguedad_promedio_meses=("antiguedad_meses", "mean"),
              antiguedad_mediana_meses=("antiguedad_meses", "median"))
         .round(2).reset_index())
tablas["P4b_comparacion_cargos"] = p4b
p4b

,abandono,clientes,cargo_mensual_promedio,cargo_mensual_mediana,cargo_total_promedio,cargo_total_mediana,antiguedad_promedio_meses,antiguedad_mediana_meses
0,No,5174,61.27,64.43,2549.91,1679.52,37.57,38.0
1,Sí,1869,74.44,79.65,1531.80,703.55,17.98,10.0


**Respuesta P4:** sí. Los clientes que abandonaron pagan **más cada mes**: su promedio es de 74,44 (mediana 79,65), frente a 61,27
(mediana 64,43) de los que permanecen. Por cuartiles, la tasa pasa del **11,24 % en el cargo bajo** (Q1) al **37,51 % en el medio-alto** (Q3)
y al 32,88 % en el alto (Q4). Q3 y Q4 suman el **79,34 % del ingreso mensual perdido**.
En cambio, su **cargo total acumulado es menor** (promedio 1 531,80 frente a 2 549,91). Esto es coherente con P3: llevan menos
tiempo como clientes (mediana de 10 meses frente a 38).

**Utilidad:** el abandono de los clientes de cargo medio-alto y alto es el que más ingreso le cuesta a la compañía.
**Limitación:** sin datos de costos no se puede decir qué clientes son más *rentables*; solo cuánto facturan. La moneda no está documentada.

### P5. ¿Hay diferencias según el método de pago y el servicio de internet?

In [23]:
# P5a: por método de pago, ordenado de mayor a menor tasa de abandono.
p5a = tabla_abandono(df, "metodo_pago").sort_values("tasa_abandono_pct", ascending=False)
tablas["P5a_abandono_por_metodo_pago"] = p5a
p5a

,metodo_pago,clientes,abandonos,tasa_abandono_pct,participacion_clientes_pct,participacion_abandonos_pct,cargo_mensual_promedio,ingreso_mensual_perdido,participacion_ingreso_perdido_pct
0,Cheque electrónico,2365,1071,45.29,33.58,57.30,76.26,84288.75,60.58
1,Cheque por correo,1612,308,19.11,22.89,16.48,43.92,16803.60,12.08
3,Transferencia bancaria (automática),1544,258,16.71,21.92,13.80,67.19,20091.90,14.44
2,Tarjeta de crédito (automática),1522,232,15.24,21.61,12.41,66.51,17946.60,12.90


In [24]:
# Resumen adicional: pago automático frente a no automático.
tabla_abandono(df, "pago_automatico")

,pago_automatico,clientes,abandonos,tasa_abandono_pct,participacion_clientes_pct,participacion_abandonos_pct,cargo_mensual_promedio,ingreso_mensual_perdido,participacion_ingreso_perdido_pct
0,No,3977,1379,34.67,56.47,73.78,63.15,101092.35,72.66
1,Sí,3066,490,15.98,43.53,26.22,66.85,38038.50,27.34


In [25]:
# P5b: por tipo de servicio de internet.
p5b = tabla_abandono(df, "servicio_internet").sort_values("tasa_abandono_pct", ascending=False)
tablas["P5b_abandono_por_internet"] = p5b
p5b

,servicio_internet,clientes,abandonos,tasa_abandono_pct,participacion_clientes_pct,participacion_abandonos_pct,cargo_mensual_promedio,ingreso_mensual_perdido,participacion_ingreso_perdido_pct
1,Fibra óptica,3096,1297,41.89,43.96,69.40,91.50,114300.05,82.15
0,DSL,2421,459,18.96,34.37,24.56,58.10,22529.20,16.19
2,Sin internet,1526,113,7.40,21.67,6.05,21.08,2301.60,1.65


**Respuesta P5:**
- **Método de pago:** el **cheque electrónico** tiene la tasa más alta (**45,29 %**). Representa el 33,58 % de los clientes, pero aporta
  el 57,30 % de los abandonos y el 60,58 % del ingreso perdido. Los pagos automáticos (transferencia 16,71 %, tarjeta 15,24 %)
  y el cheque por correo (19,11 %) están muy por debajo. Agrupando, el abandono con pago automático (15,98 %) es menos de la
  mitad que sin pago automático (34,67 %).
- **Servicio de internet:** la **fibra óptica** tiene una tasa del **41,89 %**, frente al 18,96 % de DSL y el 7,40 % de quienes no tienen
  internet. La fibra aporta el 69,40 % de los abandonos y el 82,15 % del ingreso perdido; también es el servicio con el
  cargo mensual promedio más alto (91,50).

**Utilidad:** promover el pago automático y revisar la experiencia de los clientes de fibra óptica son dos frentes concretos.
**Limitación:** el dataset no dice *por qué* abandonan los clientes de fibra (precio, calidad, competencia). Requeriría
información adicional (encuestas, reclamos) que no está disponible.

### P6. ¿Qué segmentos (contrato × internet) concentran el abandono?

In [26]:
# Una fila por combinación de contrato e internet, ordenada de mayor a menor tasa.
p6 = (tabla_abandono(df, ["tipo_contrato", "servicio_internet"])
        .sort_values("tasa_abandono_pct", ascending=False)
        .reset_index(drop=True))
p6.insert(0, "ranking", range(1, len(p6) + 1))
tablas["P6_ranking_segmentos"] = p6
p6

,ranking,tipo_contrato,servicio_internet,clientes,abandonos,tasa_abandono_pct,participacion_clientes_pct,participacion_abandonos_pct,cargo_mensual_promedio,ingreso_mensual_perdido,participacion_ingreso_perdido_pct
0,1,Mes a mes,Fibra óptica,2128,1162,54.61,30.21,62.17,87.02,100482.00,72.22
1,2,Mes a mes,DSL,1223,394,32.22,17.36,21.08,50.22,18367.25,13.20
2,3,Un año,Fibra óptica,539,104,19.29,7.65,5.56,98.78,10571.95,7.60
3,4,Mes a mes,Sin internet,524,99,18.89,7.44,5.30,20.41,1997.85,1.44
4,5,Un año,DSL,570,53,9.30,8.09,2.84,61.40,3356.25,2.41
5,6,Dos años,Fibra óptica,429,31,7.23,6.09,1.66,104.57,3246.10,2.33
6,7,Un año,Sin internet,364,9,2.47,5.17,0.48,20.82,190.25,0.14
7,8,Dos años,DSL,628,12,1.91,8.92,0.64,70.46,805.70,0.58
8,9,Dos años,Sin internet,638,5,0.78,9.06,0.27,21.78,113.50,0.08


**Respuesta P6:** el segmento **«Mes a mes | Fibra óptica»** es el primero del ranking. Tiene una tasa del **54,61 %** (más de la mitad
de sus clientes abandonó), reúne el 30,21 % de los clientes y concentra el **62,17 % de los abandonos** y el **72,22 % del ingreso
mensual perdido**. Le sigue «Mes a mes | DSL» (32,22 %). Los segmentos con contrato de dos años tienen tasas inferiores al 8 %.

**Utilidad:** con un solo segmento, el equipo de retención puede llegar a casi dos tercios de los abandonos. Es el candidato
natural para una campaña piloto (por ejemplo, migración a contrato anual con beneficio en fibra).
**Limitación:** el ranking combina solo dos variables; los segmentos pequeños («Dos años | Sin internet», 5 abandonos) tienen
tasas poco estables. Las gráficas de AE4 permitirán verlo con más detalle.

## 7. Verificar la consistencia de los resultados

In [27]:
total_clientes = len(df)
total_abandonos = int(df["abandono_num"].sum())
total_perdido = round(df["ingreso_mensual_perdido"].sum(), 2)

# En cada tabla por grupo, los clientes, abandonos e ingreso perdido deben sumar el total
# y las participaciones deben sumar 100 %.
for nombre, t in tablas.items():
    if "participacion_clientes_pct" not in t:
        continue
    assert t["clientes"].sum() == total_clientes, nombre
    assert t["abandonos"].sum() == total_abandonos, nombre
    assert abs(t["ingreso_mensual_perdido"].sum() - total_perdido) < 0.1, nombre
    for col in ["participacion_clientes_pct", "participacion_abandonos_pct", "participacion_ingreso_perdido_pct"]:
        assert abs(t[col].sum() - 100) < 0.1, (nombre, col)
    print(f"✔ {nombre}: suma {t['clientes'].sum()} clientes, {t['abandonos'].sum()} abandonos y 100 % en participaciones")

# P1 y P4b deben coincidir con los totales.
assert p1.loc[0, "abandonos"] == total_abandonos
assert p4b["clientes"].sum() == total_clientes
print("✔ P1 y P4b coinciden con los totales")

✔ P2_abandono_por_contrato: suma 7043 clientes, 1869 abandonos y 100 % en participaciones
✔ P3_abandono_por_antiguedad: suma 7043 clientes, 1869 abandonos y 100 % en participaciones
✔ P4a_abandono_por_cargo_mensual: suma 7043 clientes, 1869 abandonos y 100 % en participaciones
✔ P5a_abandono_por_metodo_pago: suma 7043 clientes, 1869 abandonos y 100 % en participaciones
✔ P5b_abandono_por_internet: suma 7043 clientes, 1869 abandonos y 100 % en participaciones
✔ P6_ranking_segmentos: suma 7043 clientes, 1869 abandonos y 100 % en participaciones
✔ P1 y P4b coinciden con los totales


## 8. Exportar el dataset preparado, las tablas y la bitácora (.csv)

In [28]:
# Dataset preparado: utf-8-sig para que Excel muestre bien las tildes.
ruta_dataset = CARPETA_DATOS / "churn_preparado.csv"
df.to_csv(ruta_dataset, index=False, encoding="utf-8-sig")

# Una tabla .csv por pregunta, más la bitácora de transformaciones.
tablas_exportar = dict(tablas, bitacora_transformaciones=tabla_bitacora)
for nombre, t in tablas_exportar.items():
    t.to_csv(CARPETA_TABLAS / f"{nombre}.csv", index=False, encoding="utf-8-sig")

print("Exportados:")
print(" -", ruta_dataset.relative_to(ROOT).as_posix())
for nombre in tablas_exportar:
    print(" -", (CARPETA_TABLAS / f"{nombre}.csv").relative_to(ROOT).as_posix())

Exportados:
 - AE3/datos/churn_preparado.csv
 - AE3/resultados/tablas/P1_resumen_general.csv
 - AE3/resultados/tablas/P2_abandono_por_contrato.csv
 - AE3/resultados/tablas/P3_abandono_por_antiguedad.csv
 - AE3/resultados/tablas/P4a_abandono_por_cargo_mensual.csv
 - AE3/resultados/tablas/P4b_comparacion_cargos.csv
 - AE3/resultados/tablas/P5a_abandono_por_metodo_pago.csv
 - AE3/resultados/tablas/P5b_abandono_por_internet.csv
 - AE3/resultados/tablas/P6_ranking_segmentos.csv
 - AE3/resultados/tablas/bitacora_transformaciones.csv


In [29]:
# Comprobamos que lo exportado coincide con lo que hay en el notebook (se leen de nuevo los archivos).
releido = pd.read_csv(ruta_dataset, encoding="utf-8-sig")
assert releido.shape == df.shape
assert releido["abandono_num"].sum() == total_abandonos
assert abs(releido["cargo_total"].sum() - df["cargo_total"].sum()) < 0.1

for nombre, t in tablas_exportar.items():
    archivo = pd.read_csv(CARPETA_TABLAS / f"{nombre}.csv", encoding="utf-8-sig")
    pd.testing.assert_frame_equal(archivo, t.reset_index(drop=True).astype(archivo.dtypes.to_dict()),
                                  check_dtype=False, check_categorical=False)
print(f"✔ Dataset ({releido.shape[0]} x {releido.shape[1]}) y {len(tablas_exportar)} tablas coinciden con el notebook")

✔ Dataset (7043 x 28) y 9 tablas coinciden con el notebook


## 9. Generar el resumen descriptivo `.txt` del DataFrame final

Como en Triple A, la clase `Reportes` genera el resumen a partir del **DataFrame final** (transformado y enriquecido):
filas, columnas, nombres y tipos, nulos, estadísticas de las columnas numéricas y frecuencias de las categóricas.
El archivo se regenera en cada ejecución.

In [30]:
origen = loader.info()
origen["muestra"] = "no (split completo); DataFrame transformado y enriquecido en AE3"
reportes = Reportes(origen=origen, incluir_categoricas=True)
ruta_resumen = reportes.guardar_reporte(df, ROOT / "AE3" / "resultados" / "resumen_dataframe_final.txt")
print("Resumen guardado en:", ruta_resumen.relative_to(ROOT).as_posix())

Resumen guardado en: AE3/resultados/resumen_dataframe_final.txt


In [31]:
# Leemos el archivo y comprobamos que corresponde a esta ejecución.
contenido = ruta_resumen.read_text(encoding="utf-8")
assert f"- Filas: {len(df)}" in contenido
assert f"- Columnas: {len(df.columns)}" in contenido
assert all(columna in contenido for columna in df.columns)
print(contenido)

RESUMEN DEL DATAFRAME
Generado: 2026-11-22 22:11:19

Origen de los datos:
- Dataset (Hugging Face): scikit-learn/churn-prediction
- Configuración: por defecto
- Split: train
- Muestra: no (split completo); DataFrame transformado y enriquecido en AE3

Dimensiones:
- Filas: 7043
- Columnas: 28
- Filas duplicadas: 0

Columnas, tipos y valores nulos:
                               tipo  nulos  valores_unicos
columna                                                   
id_cliente                      str      0            7043
adulto_mayor               category      0               2
tiene_pareja               category      0               2
tiene_dependientes         category      0               2
antiguedad_meses              int64      0              73
servicio_telefonico        category      0               2
lineas_multiples           category      0               3
servicio_internet          category      0               3
seguridad_en_linea         category      0               3
re

## 10. Conclusiones del avance

- La transformación dejó un dataset **limpio y trazable** de **7 043 clientes × 28 columnas**:
  - `TotalCharges` quedó numérico, con los 11 vacíos tratados y marcados en `cargo_total_imputado`;
  - las categorías están normalizadas en español;
  - se agregaron 7 variables derivadas.
  Todos los controles quedaron registrados en la bitácora.
- El análisis responde la necesidad: **el abandono (26,54 %) se concentra en clientes con contrato mes a mes, en su primer año,
  con fibra óptica, pago por cheque electrónico y cargos mensuales medio-altos**. El segmento «Mes a mes | Fibra óptica» por sí
  solo reúne el 62,17 % de los abandonos y el 72,22 % del ingreso mensual perdido.
- **Queda sin resolver:**
  - las causas del abandono (el análisis es descriptivo);
  - la evolución en el tiempo (no hay fechas);
  - la rentabilidad (no hay costos);
  - la aplicabilidad a una empresa real, porque los datos son ficticios.
- Las tablas exportadas en `AE3/resultados/tablas/` quedan listas para la presentación gráfica de la AE4.